# Itemset — native Julia on Colab

Select **Runtime → Change runtime type → Julia**. All code cells run directly in the Julia kernel. Use Julia 1.12+ within 1.x; if needed select runtime version **2026.07** (Julia 1.12.6) or a newer compatible runtime. See [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html).

For this local repository, export a committed ZIP with `git archive --format=zip --output=../Itemset-colab.zip HEAD`, upload it through Colab's **Files** sidebar to `/content/Itemset-colab.zip`, and run all cells. Alternatively set a future public GitHub URL and exact commit below.

The notebook verifies the environment and tests, then runs the Semeion image pilot. Keep the ZIP revision with the output.

In [ ]:
REPO_URL = ""  # Optional future public GitHub HTTPS URL.
REVISION = ""  # Exact commit required for cloning; record the ZIP's commit too.
ROOT = get(ENV, "ITEMSET_ROOT", isdir("/content") ? "/content/Itemset" : pwd())
ARCHIVE = "/content/Itemset-colab.zip"  # Upload through Colab's Files sidebar.

if !isempty(REPO_URL)
    isempty(REVISION) && error("Set an exact commit in REVISION for reproducibility")
    isdir(joinpath(ROOT, ".git")) || run(`git clone $REPO_URL $ROOT`)
    run(`git -C $ROOT checkout --detach $REVISION`)
elseif !isfile(joinpath(ROOT, "Project.toml"))
    isfile(ARCHIVE) || error("Upload Itemset-colab.zip through the Files sidebar first")
    mkpath(ROOT)
    run(`unzip -q -o $ARCHIVE -d $ROOT`)
end

isfile(joinpath(ROOT, "Project.toml")) || error("Repository root must contain Project.toml")
ROOT = abspath(ROOT)
cd(ROOT)
println("Repository: ", ROOT)
println("Revision: ", isdir(".git") ? readchomp(`git rev-parse HEAD`) :
        isempty(REVISION) ? "local archive; record its commit before final experiments" : REVISION)


In [ ]:
using Pkg
v"1.12" <= VERSION < v"2" ||
    error("Select a Colab Julia 1.12+ runtime (e.g. runtime version 2026.07)")
Pkg.activate(ROOT)
Pkg.instantiate()

using ItemsetMining
using Random
println("Julia: ", VERSION, " | threads: ", Threads.nthreads(), " | seed: ", DEFAULT_SEED)
println("Project: ", Base.active_project())


In [ ]:
# Execute setup and tests directly inside this Julia kernel.
# Separate modules keep test definitions out of the notebook workspace.
Base.include(Module(:ItemsetSetupCheck), joinpath(ROOT, "scripts", "check_setup.jl"))
module ItemsetSetupTests
    include(joinpath(Main.ROOT, "test", "runtests.jl"))
end


In [ ]:
transactions = read_transactions(joinpath(ROOT, "data", "toy", "baskets.txt"))
minsup_count = support_threshold(0.5, length(transactions))
(; transactions, minsup_count)


## Hung — Eclat, tests, scalability, output size

`mine` with `variant=:baseline` runs the paper-based Eclat described in `docs/eclat.md`. The cell below mines the toy baskets at minsup 0.5 and prints the SPMF output, which matches `data/toy/baskets.expected.txt`. The test cell above checks the baseline against every toy fixture and a brute-force count.

In [ ]:
patterns = mine(transactions; minsup=0.5, variant=:baseline)
write_itemsets(stdout, patterns)

## Thien — Market baskets, runtime, transaction width

Add public labelled retail data acquisition, checksum/license, cleaning, reversible mappings, own Eclat invocation, rule tests, top-10 lift rules and interpretation (DAT-10/14/37). Add the common benchmark runner and runtime/width plots. Pin approved dependencies and record seed 20261018.

## Phat — Semeion image pilot

The [UCI Semeion dataset](https://archive.ics.uci.edu/dataset/178/semeion+handwritten+digit) contains binary 16×16 handwritten digits under CC BY 4.0. Download the original text file, verify its SHA-256, and run the fixed-seed 4×4 patch encoder. The codebook has 24 words. The baseline miner counts visual-word itemsets at three support thresholds. The SVGs show source digits and actual observed patches. Record the runtime and revision with the outputs.

In [ ]:
using Downloads, SHA
semeion_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/semeion/semeion.data"
semeion_sha256 = "f43228ae3da5ea6a3c95069d53450b86166770e3b719dcc333182128fe08d4b1"
semeion_path = joinpath(ROOT, "data", "raw", "semeion", "semeion.data")
mkpath(dirname(semeion_path))
isfile(semeion_path) || Downloads.download(semeion_url, semeion_path)
bytes2hex(sha256(read(semeion_path))) == semeion_sha256 || error("Semeion checksum mismatch")
pilot_output = joinpath(mktempdir(), "semeion-pilot")
module ImagePilotRun
    include(joinpath(Main.ROOT, "scripts", "image_pilot.jl"))
end
ImagePilotRun.main([semeion_path, pilot_output])
println(read(joinpath(pilot_output, "thresholds.tsv"), String))
display(MIME("image/svg+xml"), read(joinpath(pilot_output, "digits.svg"), String))
display(MIME("image/svg+xml"), read(joinpath(pilot_output, "visual_words.svg"), String))


## Clean-run checklist

Restart the Julia runtime and run all cells after modules are implemented. Record the Colab runtime version, Julia version, thread count, revision, dataset hashes, dependencies, parameters, stdout, and failures. Publish a public Colab link in README after verifying access and a complete clean run. A local run does not prove the hosted Colab runtime has been exercised.